### MultiQueryRetriever

`MultiQueryRetriever` asks an LLM to rewrite one question into several search queries, retrieves for each query, and returns the unique combined documents. It is useful when the original question is ambiguous or uses different wording from the source documents.

In [67]:
import os

from dotenv import load_dotenv
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint

load_dotenv()
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    raise ValueError("Set HF_TOKEN or HUGGINGFACEHUB_API_TOKEN in .env")

In [68]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(
    documents=all_docs, 
    embedding=embeddings
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8128.19it/s]


In [69]:
# Simple basic similarity search retriever
similarity_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={"k": 5}
    )

In [70]:
# Featherless serves this model through its chat-completion (conversational) API.
endpoint = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    provider="featherless-ai",
    huggingfacehub_api_token=hf_token,
    max_new_tokens=256,
    temperature=0,
    do_sample=False,
)

# This wrapper calls InferenceClient.chat_completion(), not text_generation().
llm = ChatHuggingFace(llm=endpoint)

#### Creating Multi Query  Retriever

In [71]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever= vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
    # include_original=True,  # search the user's exact query too
)

In [72]:
query = "How to improve Energy levels and maintain balance?"


In [73]:
# multi_query_retriever = MultiQueryRetriever.from_llm(
#     retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
#     llm=llm,
# )

multi_query_results = multi_query_retriever.invoke(query)

#### Getting the result of both of the retriver and comparing them

In [74]:
similarity_results = similarity_retriever.invoke(query)
# multi_query_results = multi_query_retriever.invoke(query)

In [75]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multi_query_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Python balances readability with power, making it a popular system design language.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
The solar energy system in modern homes helps balance electri

#### Important

With LangChain 1.x, import this retriever from `langchain_classic.retrievers.multi_query`, not `langchain.retrievers.multi_query`. Run the cells in order. The first embedding run may download the embedding model.